I ran cell ranger aggr on all my samples, but for geo submission, it requires one processed data file per sample, so I am splitting the count matrix by sample. 

In [1]:
import pandas as pd
import scanpy as sc
import gzip
h5ad_file = "/home/catherine/phd/projects/termites/data/znev/combined_no_norm_clustered.h5ad"
adata = sc.read_h5ad(h5ad_file)

In [17]:
## to save h5ad file for geo uplaod 


h5ad_file = "/home/catherine/phd/projects/termites/data/znev/combined_no_norm_clustered.h5ad"
adata_geo = sc.read_h5ad(h5ad_file)

columns_to_drop = ['samap_annot', 'labels_only_znev_fabio', 'leiden_fabio', 'leiden_dmel_fabio', 'comparison_group']
adata_geo.obs = adata_geo.obs.drop(columns=columns_to_drop)

# Renaming 'old_name' to 'new_name' in observation metadata
adata_geo.obs.rename(columns={'znev_leiden_dmel_fabio': 'saturn_dmel_annotation'}, inplace=True)

results_file = "/home/catherine/phd/projects/termites/data/znev/znev_all_castes.h5ad"
adata_geo.write(results_file)

In [15]:
# first I need to check column names etc 

adata_geo.obs

,sample#,caste,n_genes_by_counts,total_counts,leiden,saturn_dmel_annotation,paper_cell_type_annotation
AAACCCATCACATACG-1,1,king,546,742.0,15,tracheocyte,T5
AAACCCATCGACCAAT-1,1,king,762,1052.0,0,epithelial,epithelial
AAACGAAAGGCCACCT-1,1,king,430,571.0,0,epithelial,epithelial
AAACGAACACTTCAAG-1,1,king,785,1121.0,2,stem,stem cell
AAACGAATCCATTTAC-1,1,king,575,743.0,11,tormogen,sensory neuron
...,...,...,...,...,...,...,...
TTTGTTGAGTGATAAC-4,4,worker,439,543.0,4,tracheocyte,T3
TTTGTTGCACGACAGA-4,4,worker,1627,2953.0,21,cyst,intestinal progenitor like
TTTGTTGCAGCATACT-4,4,worker,592,773.0,4,tracheocyte,T3
TTTGTTGTCATAGGCT-4,4,worker,498,750.0,12,muscle,muscle cell


In [6]:
adata_worker   = adata[adata.obs["caste"] == "worker"].copy()
adata_soldier  = adata[adata.obs["caste"] == "soldier"].copy()
adata_king     = adata[adata.obs["caste"] == "king"].copy()
adata_queen    = adata[adata.obs["caste"] == "queen"].copy()

In [10]:
matrix = "/home/catherine/phd/projects/termites/data/oist_matrices/znev_castes_combined_craggr_no_norm/matrix.mtx.gz"

In [9]:
caste_num = {
         '1': 'king',
         '2': 'queen',
         '3': 'soldier',
         '4': 'worker',
     }

In [13]:
import gzip
import itertools

with gzip.open(matrix, 'rt') as f:
    for line in itertools.islice(f, 10):
        print(line.strip())

%%MatrixMarket matrix coordinate integer general
%metadata_json: {"software_version": "cellranger-7.2.0", "format_version": 2}
14272 24252 19757189
22 1 1
23 1 3
28 1 2
60 1 1
73 1 1
120 1 2
128 1 1


In [27]:
import scipy as sp
import pandas as pd

matrix_fn = "/home/catherine/phd/projects/termites/data/oist_matrices/znev_castes_combined_craggr_no_norm/matrix.mtx.gz"
barcodes_fn = "/home/catherine/phd/projects/termites/data/oist_matrices/znev_castes_combined_craggr_no_norm/barcodes.tsv.gz"

caste_num = {
         '1': 'king',
         '2': 'queen',
         '3': 'soldier',
         '4': 'worker',
     }

import gzip
with gzip.open(matrix_fn, "rb") as mfile, gzip.open(barcodes_fn, "rb") as bfile:
    matrix = sp.io.mmread(mfile).tocsc()
    barcodes = pd.read_csv(bfile, header=None)
    barcodes["sample_number"] = [x[-1] for x in barcodes[0].values]
    barcodes["caste"] = barcodes["sample_number"].map(caste_num)

    for caste, barcodes_caste in barcodes.groupby("caste"):
        idx_rows = barcodes_caste.index.values
        matrix_caste = matrix[:, idx_rows].tocoo()
        barcodes_caste = barcodes_caste[0]

        matrix_fn_caste = matrix_fn.split(".")[0] + "_" + caste + ".mtx.gz"
        barcodes_fn_caste = barcodes_fn.split(".")[0] + "_" + caste + ".tsv.gz"

        with gzip.open(matrix_fn_caste, "wb") as mout, gzip.open(barcodes_fn_caste, "wb") as bout:
            # TODO: write barcodes_caste and matrix_caste to the 2 caste-specific files
            barcodes_caste.to_csv(bout, index=False)
            sp.io.mmwrite(mout, matrix_caste)

In [29]:
!ls /home/catherine/phd/projects/termites/data/oist_matrices/znev_castes_combined_craggr_no_norm/

barcodes_king.tsv.gz	 barcodes_worker.tsv.gz  matrix_queen.mtx.gz
barcodes_queen.tsv.gz	 features.tsv.gz	 matrix_soldier.mtx.gz
barcodes_soldier.tsv.gz  matrix_king.mtx.gz	 matrix_worker.mtx.gz
barcodes.tsv.gz		 matrix.mtx.gz


In [30]:
adata

AnnData object with n_obs × n_vars = 24252 × 14272
    obs: 'sample#', 'caste', 'n_genes_by_counts', 'total_counts', 'leiden', 'samap_annot', 'labels_only_znev_fabio', 'leiden_fabio', 'leiden_dmel_fabio', 'znev_leiden_dmel_fabio', 'paper_cell_type_annotation', 'comparison_group'
    var: 'gene_ids', 'feature_types', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'dmel_gene_ortho', 'dmel_gene_symbol'
    uns: 'annotate_cells_colors', 'annotations_1_colors', 'caste_colors', 'hvg', 'leiden', 'leiden_colors', 'leiden_dmel_fabio_colors', 'leiden_fabio_colors', 'log1p', 'neighbors', 'new_clusters_colors', 'paper_cell_type_annotation_colors', 'pca', 'rank_genes_groups', 'samap_annot_colors', 'umap', 'znev_leiden_dmel_fabio_colors'
    obsm: 'X_pca', 'X_umap', 'X_umap_fabio'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'